# Assignment 4: Building Walking Catchment Areas

In the **fourth module** we met the tools of network analysis and learned to build routes, distance matrices and service areas, using both a local graph and external APIs.

In this assignment you will:

1. build 5, 10 and 15-minute walking catchment areas (isochrones) for one category of features in your study area;
2. wrap the calculation into a function;
3. apply that function to the other categories of features prepared in assignment 1.

Use the data you prepared in the first assignment as your input.

> The code cells below are placeholders for your own work: they are left unexecuted on purpose. Fill them in as you go, and run them in your own copy of the notebook.


## Suggested Plan of Work

### Step 0. Importing Libraries


In [ ]:
# your code


### Step 1. Preparing the Data


Choose **one category of features** from the first assignment and load its layer from `project.gpkg` — `poi_grocery`, for example. If you did not save it, download it again: see step 2 of [assignment 1](../module_1/geopy_task1.ipynb).

Start with a category that has a moderate number of features. Every feature needs its own subgraph for every time interval, so a category with 200 points and three intervals means 600 calls — noticeably slower than a category with 20.

In [ ]:
# your code

# poi = gpd.read_file("project.gpkg", layer="poi_grocery")

If the dataset mixes geometry types — points and polygons, say — bring everything to a single type. We recommend a point representation.

Think about how to convert polygons to points correctly (centroids, for example) so that no features drop out of the analysis. Remember that geometric operations belong in a projected CRS, as we saw in the second module.


In [ ]:
# your code


### Step 2. Building the Catchment Areas

At this stage you build the catchment areas **on a local street network graph**, using `osmnx` and `networkx`.

In the module materials the service area was built from a **distance**. Here you need catchment areas defined by **travel time** — 5, 10 and 15 minutes.

To do that you need to:

- decide on an average walking speed;
- convert each time interval into the distance a pedestrian covers in that time;
- use those distances as the limit when building the catchment areas on the graph.

**Use 5 km/h — about 83 metres per minute — unless you have a reason not to.** That is the value normally assumed for walking accessibility studies, and fixing it means your results stay comparable with everyone else's. If you change it, say so when you interpret the results: at 4 km/h a "15-minute city" shrinks by a fifth.

#### 2.1. Limitations of the Method

When working with a local street network graph, keep the following in mind:

- OpenStreetMap data quality varies from place to place;
- not every edge of the graph carries the attributes needed to calculate travel time;
- calculations on large graphs can take a long time;
- the catchment area is limited by the extent of the graph you downloaded.

Briefly describe which of these — or which others — you ran into while doing the assignment.

Doing this now helps you avoid mistakes while building the zones, and interpret the results correctly afterwards.


> _your notes here_


#### 2.2. Preparing the Graph

Prepare the street network graph for the calculation.

Download the graph for your study area (with `osmnx`) and look at its properties. Pay attention to the `network_type` argument of `ox.graph_from_place()` and decide which network type you need for a walking analysis.

Download it **once** and reuse the same graph object for every category and every time interval — re-downloading it in a loop is the most common reason this assignment takes hours instead of minutes.

In [ ]:
# your code


#### 2.3. Finding the Nearest Graph Node for Each Feature

To calculate anything on the graph, your features have to be matched to the network's nodes.

For each point, find the nearest graph node with `ox.distance.nearest_nodes()` and store the correspondence between features and nodes.

_Hint:_ you have already found the nearest node for a single point. For a `GeoDataFrame` the same call accepts whole coordinate columns at once.


In [ ]:
# Hint (one possible approach)

# nodes = ox.distance.nearest_nodes(
#     graph,
#     X=gdf.geometry.x,
#     Y=gdf.geometry.y
# )

# the resulting list of nodes can be stored as a new column:
# gdf["node"] = nodes


#### 2.4. Building the Subgraphs

Build **5, 10 and 15-minute** walking catchment areas from your features.

`nx.ego_graph()` limits the subgraph by **distance** along the graph edges, so for each time interval you first need the matching distance, calculated from your average walking speed.

In the module materials the service area was built for **one point**. Now you have a set of nodes (see step 2.3), so the operation has to be repeated for each of them.


In [ ]:
# Hint (one possible approach)

# distance for the catchment areas
# max_dist_5min =


# example: 5-minute catchment areas for a set of points
# subgraphs_5min = []

# for node in gdf["node"]:
#     sub = nx.ego_graph(graph, node, radius=max_dist_5min, distance="length")
#     subgraphs_5min.append(sub)


> Repeat this for each time interval.


#### 2.5. Converting the Subgraphs into Zones

A subgraph is a set of edges, so to go further you need to turn it into geometry.

For each subgraph:

- convert it into a `GeoDataFrame` of edges;
- merge those edges into a single geometry;
- build a polygon around them with `convex_hull`.

Keep in mind that such a polygon is an approximation of the real catchment area.


In [ ]:
# Hint (one possible approach)

# polygons_5min = []

# for subgraph in subgraphs_5min:
#     edges = ox.graph_to_gdfs(subgraph, nodes=False, edges=True)
#     geom = edges.geometry.union_all()
#     polygon = geom.convex_hull
#     polygons_5min.append(polygon)


> Repeat this for each time interval.


#### 2.6. Merging the Catchment Areas


Create the final catchment area for each time interval:

- 5 minutes;
- 10 minutes;
- 15 minutes.

`dissolve()` is one way to do it.


In [ ]:
# Hint (one possible approach)

# a GeoDataFrame with the individual 5-minute zones
# iso_5min_gdf = gpd.GeoDataFrame(geometry=polygons_5min, crs=gdf.crs)

# merging them into one final zone
# iso_5min_union = iso_5min_gdf.dissolve()


> Repeat this for each time interval.


### Step 3. Visualisation


Show on a map:

- the features you chose;
- the 5, 10 and 15-minute catchment areas;
- the boundary of the study area, if it helps.


In [ ]:
# your code


### Step 4. Analysing the Results


Analyse what you have and answer the questions:

- Are there parts of the territory that stay poorly served?
- Do the results match what you expected?
- Which limitations of the method should be kept in mind when interpreting them?


> _your notes here_


### Step 5. Saving the Data


Save the catchment areas into `project.gpkg`, following the layout agreed in [assignment 1](../module_1/geopy_task1.ipynb):

- `iso_<category>` — the merged zones, **one row per time interval**, with the interval in a column called `minutes` (5, 10, 15);
- `iso_<category>_by_feature` — the individual zones, one per feature, if you want to keep them.

The `minutes` column matters: [assignment 5](../module_5/geopy_task5.ipynb) groups the population figures by it, and [assignment 6](../module_6/geopy_task6.ipynb) styles the zones by it. Save in EPSG:4326, as agreed.

In the next assignment we calculate the share of the population that falls outside these zones.

In [ ]:
# your code

# iso_grocery.to_crs("EPSG:4326").to_file("project.gpkg", layer="iso_grocery", driver="GPKG")

### Step 6. Automating the Calculation

Now that you have built the catchment areas for one category, repeat the same algorithm for the other categories prepared in assignment 1.

To avoid duplicating code, wrap the main steps into a function.

The function should:

1. take as input:
   - a `GeoDataFrame` with the features of one category;
   - the walking network graph;
   - the name of the category;
   - a list of time intervals, for example `(5, 10, 15)`;

2. convert the feature geometries to points;

3. find the nearest graph node for every feature;

4. for each time interval:
   - calculate the maximum distance;
   - build the subgraph around every feature;
   - convert the subgraphs into polygons;
   - create the individual catchment zones;
   - merge them into one final zone;

5. return or save the results:
   - the individual catchment zones per feature;
   - the merged catchment area for the whole category.

> Try to keep the function general enough to work with any category of features without changing its body.


An outline of the function:


In [ ]:
# def build_isochrones_for_category(
#     gdf,
#     graph,
#     category_name,
#     time_intervals=(5, 10, 15)
# ):
#     """
#     Builds walking catchment areas for one category of features.
#
#     The function should:
#     1. convert the geometries to points;
#     2. find the nearest graph nodes;
#     3. calculate the distances for the given time intervals;
#     4. build the catchment subgraphs;
#     5. convert the subgraphs into polygons;
#     6. create the individual and the merged catchment zones.
#     """
#
#     # your code
#
#     return individual_zones, dissolved_zones


Test the function on the category you have already built the zones for — the results should match.


In [ ]:
# your code


Then apply it to every category downloaded in assignment 1:


In [ ]:
# Example
# all_individual_zones = {}
# all_dissolved_zones = {}

# for category_name, category_gdf in categories.items():
#     individual, dissolved = build_isochrones_for_category(
#         gdf=category_gdf,
#         graph=graph,
#         category_name=category_name,
#         time_intervals=(5, 10, 15)
#     )

#     all_individual_zones[category_name] = individual
#     all_dissolved_zones[category_name] = dissolved


Visualise the results to check them.


In [ ]:
# your code


Save the data.


In [ ]:
# your code


### Step 7. Network Zones vs. Buffers

In [assignment 3](../module_3/geopy_task3.ipynb) you built catchment areas around the public transport stops with **buffers** — circles of a fixed radius. Here you build them along the **street network**. The two answer the same question in different ways, and it is worth seeing by how much they differ.

Run your function on the `stops` layer from `project.gpkg`, using a time interval that corresponds to the buffer radius you chose earlier (at 83 m/min, a 500-metre buffer is roughly a 6-minute walk), then compare:

- the area covered by the buffers and by the network zones;
- how many buildings fall inside each of them;
- where the difference is largest — and what in the street layout explains it.

A buffer ignores rivers, railways and dead ends, so it always overstates what is actually reachable. Seeing the size of that gap on your own data is the point of this step.

In [ ]:
# your code

> _your notes here_

## Result

By the end of this assignment you should have:

- 5, 10 and 15-minute catchment areas for one chosen category of features;
- a function that builds those zones for any other category;
- catchment areas for several categories from assignment 1;
- a visualisation of the results;
- a short assessment of the limitations of the approach;
- the results saved in `project.gpkg` as `iso_<category>` layers with a `minutes` column;
- a comparison of the network zones with the buffers from assignment 3.